In [2]:
# ==============================================================================
# VisionBench - Baseline 9: Multi-Model Anomaly Score Fusion
# Combines per-image predictions from PatchCore, PaDiM, and FastFlow
# ==============================================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from sklearn.metrics import roc_auc_score
import kagglehub

# ------------------------------------------------------------------------------
# 1. Dataset Setup & Path Configuration
# ------------------------------------------------------------------------------
print("--- Step 1: Loading MVTec AD Dataset ---")
kaggle_path = kagglehub.dataset_download("ipythonx/mvtec-ad")
DATA_DIR = Path(kaggle_path)
RESULTS_DIR = Path("./results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CATEGORIES = [
    "bottle", "cable", "capsule", "carpet", "grid",
    "hazelnut", "leather", "metal_nut", "pill", "screw",
    "tile", "toothbrush", "transistor", "wood", "zipper"
]

class MVTecTestDataset(Dataset):
    def __init__(self, root_dir, category, transform=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []

        cat_path = root_dir / category / "test"
        for sub_dir in cat_path.iterdir():
            if sub_dir.is_dir():
                is_good = (sub_dir.name == "good")
                for img in sub_dir.glob("*.png"):
                    self.image_paths.append(img)
                    self.labels.append(0 if is_good else 1)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx], str(img_path)

class MVTecTrainDataset(Dataset):
    def __init__(self, root_dir, category, transform=None):
        self.transform = transform
        self.image_paths = list((root_dir / category / "train" / "good").glob("*.png"))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image

img_transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running Baseline 9 on Device: {device}\n")

# ------------------------------------------------------------------------------
# 2. Model Architecture Definitions
# ------------------------------------------------------------------------------

# Model A: PatchCore Sub-Extractor
class PatchCoreModel:
    def __init__(self, device):
        self.device = device
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT).to(device)
        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-2]).eval()
        for param in self.feature_extractor.parameters():
            param.requires_grad = False
        self.memory_bank = None

    def fit(self, train_loader):
        features_list = []
        with torch.no_grad():
            for images in train_loader:
                images = images.to(self.device)
                feats = self.feature_extractor(images)  # [B, 512, 8, 8]
                feats = feats.permute(0, 2, 3, 1).reshape(-1, 512)
                features_list.append(feats.cpu())
        full_bank = torch.cat(features_list, dim=0)
        # Subsample memory bank (10% coreset) for fast search
        indices = torch.randperm(full_bank.size(0))[:max(1, full_bank.size(0) // 10)]
        self.memory_bank = full_bank[indices].to(self.device)

    def predict(self, test_loader):
        scores = []
        with torch.no_grad():
            for images, _, _ in test_loader:
                images = images.to(self.device)
                feats = self.feature_extractor(images)
                B, C, H, W = feats.shape
                feats = feats.permute(0, 2, 3, 1).reshape(B, H * W, C)

                # Minimum distance to memory bank per patch, then max per image
                img_scores = []
                for b in range(B):
                    dists = torch.cdist(feats[b], self.memory_bank)
                    min_dists, _ = torch.min(dists, dim=1)
                    img_scores.append(torch.max(min_dists).item())
                scores.extend(img_scores)
        return np.array(scores)

# Model B: PaDiM Sub-Extractor
class PaDiMModel:
    def __init__(self, device):
        self.device = device
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT).to(device)
        self.layer1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool, resnet.layer1).eval()
        self.layer2 = resnet.layer2.eval()
        for param in self.layer1.parameters(): param.requires_grad = False
        for param in self.layer2.parameters(): param.requires_grad = False
        self.mean = None
        self.cov_inv = None

    def fit(self, train_loader):
        feats_list = []
        with torch.no_grad():
            for images in train_loader:
                images = images.to(self.device)
                f1 = self.layer1(images)
                f2 = self.layer2(f1)
                f2_up = F.interpolate(f2, size=f1.shape[-2:], mode='bilinear', align_corners=False)
                cat_feats = torch.cat([f1, f2_up], dim=1) # [B, 192, 64, 64]
                feats_list.append(cat_feats.cpu())

        all_feats = torch.cat(feats_list, dim=0) # [N, C, H, W]
        N, C, H, W = all_feats.shape
        all_feats = all_feats.permute(0, 2, 3, 1).reshape(N, H * W, C)

        self.mean = torch.mean(all_feats, dim=0) # [HW, C]

        # Calculate covariance & inverse
        cov_inv_list = []
        for i in range(H * W):
            sub_feats = all_feats[:, i, :] - self.mean[i]
            cov = (sub_feats.T @ sub_feats) / (N - 1) + 0.01 * torch.eye(C)
            cov_inv_list.append(torch.linalg.inv(cov).unsqueeze(0))
        self.cov_inv = torch.cat(cov_inv_list, dim=0).to(self.device) # [HW, C, C]
        self.mean = self.mean.to(self.device)

    def predict(self, test_loader):
        scores = []
        with torch.no_grad():
            for images, _, _ in test_loader:
                images = images.to(self.device)
                f1 = self.layer1(images)
                f2 = self.layer2(f1)
                f2_up = F.interpolate(f2, size=f1.shape[-2:], mode='bilinear', align_corners=False)
                cat_feats = torch.cat([f1, f2_up], dim=1)
                B, C, H, W = cat_feats.shape
                cat_feats = cat_feats.permute(0, 2, 3, 1).reshape(B, H * W, C)

                img_scores = []
                for b in range(B):
                    delta = cat_feats[b] - self.mean # [HW, C]
                    # Mahalanobis distance per location
                    dist = torch.bmm(delta.unsqueeze(1), self.cov_inv).squeeze(1) # [HW, C]
                    dist = torch.sum(dist * delta, dim=1) # [HW]
                    img_scores.append(torch.max(dist).item())
                scores.extend(img_scores)
        return np.array(scores)

# Model C: FastFlow Sub-Extractor
class FlowSubstep(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels // 2, in_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels, in_channels // 2, kernel_size=3, padding=1)
        )
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return torch.cat([x1, x2 + self.net(x1)], dim=1)

class FastFlowModel(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-3])
        for param in self.feature_extractor.parameters():
            param.requires_grad = False
        self.flow = nn.Sequential(FlowSubstep(256), FlowSubstep(256), FlowSubstep(256))

    def forward(self, x):
        features = self.feature_extractor(x)
        z = self.flow(features)
        loss = 0.5 * torch.mean(z ** 2)
        return loss, z

def train_fastflow(train_loader, device, epochs=3):
    model = FastFlowModel().to(device)
    optimizer = torch.optim.Adam(model.flow.parameters(), lr=1e-3)
    for _ in range(epochs):
        model.train()
        for images in train_loader:
            images = images.to(device)
            optimizer.zero_grad()
            loss, _ = model(images)
            loss.backward()
            optimizer.step()
    return model

def predict_fastflow(model, test_loader, device):
    model.eval()
    scores = []
    with torch.no_grad():
        for images, _, _ in test_loader:
            images = images.to(device)
            _, z = model(images)
            batch_scores = torch.mean(z ** 2, dim=[1, 2, 3]).cpu().numpy()
            scores.extend(batch_scores)
    return np.array(scores)

# ------------------------------------------------------------------------------
# 3. Min-Max Normalization Function
# ------------------------------------------------------------------------------
def min_max_normalize(scores):
    min_val, max_val = np.min(scores), np.max(scores)
    if max_val - min_val < 1e-8:
        return np.zeros_like(scores)
    return (scores - min_val) / (max_val - min_val)

# ------------------------------------------------------------------------------
# 4. Multi-Category Execution Loop
# ------------------------------------------------------------------------------
all_image_records = []
category_metrics = []

print("--- Step 2: Executing Inference & Score Fusion Across All 15 Categories ---")

for cat in CATEGORIES:
    print(f"\nProcessing Category: [{cat.upper()}]")

    train_ds = MVTecTrainDataset(DATA_DIR, cat, transform=img_transform)
    test_ds = MVTecTestDataset(DATA_DIR, cat, transform=img_transform)

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

    # Extract metadata & ground truths
    labels = np.array([test_ds[i][1] for i in range(len(test_ds))])
    paths = [test_ds[i][2] for i in range(len(test_ds))]

    # 1. PatchCore Inference
    pc_model = PatchCoreModel(device)
    pc_model.fit(train_loader)
    pc_scores = pc_model.predict(test_loader)
    pc_auroc = roc_auc_score(labels, pc_scores)

    # 2. PaDiM Inference
    padim_model = PaDiMModel(device)
    padim_model.fit(train_loader)
    padim_scores = padim_model.predict(test_loader)
    padim_auroc = roc_auc_score(labels, padim_scores)

    # 3. FastFlow Inference
    ff_model = train_fastflow(train_loader, device, epochs=3)
    ff_scores = predict_fastflow(ff_model, test_loader, device)
    ff_auroc = roc_auc_score(labels, ff_scores)

    # 4. Normalization
    pc_norm = min_max_normalize(pc_scores)
    padim_norm = min_max_normalize(padim_scores)
    ff_norm = min_max_normalize(ff_scores)

    # 5. Equal-Weighted Score Fusion
    fused_scores = (1/3 * pc_norm) + (1/3 * padim_norm) + (1/3 * ff_norm)
    fusion_auroc = roc_auc_score(labels, fused_scores)

    # Store Category Metrics
    category_metrics.append({
        "category": cat,
        "PatchCore_AUROC": pc_auroc,
        "PaDiM_AUROC": padim_auroc,
        "FastFlow_AUROC": ff_auroc,
        "Fusion_AUROC": fusion_auroc
    })

    print(f"  -> PatchCore: {pc_auroc:.4f} | PaDiM: {padim_auroc:.4f} | FastFlow: {ff_auroc:.4f} | FUSION: {fusion_auroc:.4f}")

    # Store Per-Image Records
    for i in range(len(test_ds)):
        all_image_records.append({
            "category": cat,
            "image_path": paths[i],
            "label": labels[i],
            "patchcore_score": pc_scores[i],
            "padim_score": padim_scores[i],
            "fastflow_score": ff_scores[i],
            "patchcore_normalized": pc_norm[i],
            "padim_normalized": padim_norm[i],
            "fastflow_normalized": ff_norm[i],
            "fused_score": fused_scores[i]
        })

# ------------------------------------------------------------------------------
# 5. Export Results & Final Summary Matrix
# ------------------------------------------------------------------------------
df_images = pd.DataFrame(all_image_records)
df_images.to_csv(RESULTS_DIR / "baseline9_per_image_predictions.csv", index=False)

df_summary = pd.DataFrame(category_metrics)
df_summary.to_csv(RESULTS_DIR / "baseline9_category_summary.csv", index=False)

print("\n==========================================================================")
print("             VISIONBENCH: BASELINE 9 SCORE FUSION RESULTS")
print("==========================================================================")
print(f"{'Category':<15} | {'PatchCore':<10} | {'PaDiM':<10} | {'FastFlow':<10} | {'FUSION':<10}")
print("--------------------------------------------------------------------------")
for _, row in df_summary.iterrows():
    print(f"{row['category']:<15} | {row['PatchCore_AUROC']:<10.4f} | {row['PaDiM_AUROC']:<10.4f} | {row['FastFlow_AUROC']:<10.4f} | {row['Fusion_AUROC']:<10.4f}")

mean_pc = df_summary["PatchCore_AUROC"].mean()
mean_padim = df_summary["PaDiM_AUROC"].mean()
mean_ff = df_summary["FastFlow_AUROC"].mean()
mean_fusion = df_summary["Fusion_AUROC"].mean()

print("--------------------------------------------------------------------------")
print(f"{'MEAN AUROC':<15} | {mean_pc:<10.4f} | {mean_padim:<10.4f} | {mean_ff:<10.4f} | {mean_fusion:<10.4f}")
print("==========================================================================")
print(f"\nPer-image predictions saved to: {RESULTS_DIR / 'baseline9_per_image_predictions.csv'}")

--- Step 1: Loading MVTec AD Dataset ---
Using Colab cache for faster access to the 'mvtec-ad' dataset.
Running Baseline 9 on Device: cuda

--- Step 2: Executing Inference & Score Fusion Across All 15 Categories ---

Processing Category: [BOTTLE]
  -> PatchCore: 0.9937 | PaDiM: 0.9937 | FastFlow: 0.9627 | FUSION: 0.9944

Processing Category: [CABLE]
  -> PatchCore: 0.8396 | PaDiM: 0.7723 | FastFlow: 0.7022 | FUSION: 0.8735

Processing Category: [CAPSULE]
  -> PatchCore: 0.7527 | PaDiM: 0.8472 | FastFlow: 0.5082 | FUSION: 0.7587

Processing Category: [CARPET]
  -> PatchCore: 0.8860 | PaDiM: 0.9739 | FastFlow: 0.7761 | FUSION: 0.9627

Processing Category: [GRID]
  -> PatchCore: 0.4453 | PaDiM: 0.9582 | FastFlow: 0.2999 | FUSION: 0.5923

Processing Category: [HAZELNUT]
  -> PatchCore: 0.9361 | PaDiM: 0.6714 | FastFlow: 0.9557 | FUSION: 0.9457

Processing Category: [LEATHER]
  -> PatchCore: 0.9046 | PaDiM: 0.9915 | FastFlow: 0.9440 | FUSION: 0.9942

Processing Category: [METAL_NUT]
  -> Pa